In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Access Aperture4 Simulation Data

This notebook demonstrates how to access variables from Aperture4 simulation output in `/data/personal/Alex_Chen/Torus/Data_vert_B0.6_pair/`.

We use the analysis tools from `analysis_folder` and `Aperture4/external/Aperture_plotting`.

In [2]:
# Add Aperture4 python modules to path
sys.path.insert(0, '/home/aaron/Research/Aperture4/python')
sys.path.insert(0, '/home/aaron/Research/Aperture4/external/Aperture_plotting')

# Add current analysis folder to path
sys.path.insert(0, '/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder')

print("Python paths configured:")
for p in sys.path[:3]:
    print(f"  {p}")

Python paths configured:
  /home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder
  /home/aaron/Research/Aperture4/external/Aperture_plotting
  /home/aaron/Research/Aperture4/python


In [655]:
# Import the Kerr-Schild data loader from the analysis folder
import datalib_gr_ks as ks

# Also import base datalib for reference
from datalib import Data, flag_to_species

## Define Data Directory and Inspect Contents

The simulation data is stored in `/data/personal/Alex_Chen/Torus/Data_vert_B0.6_pair/`.

In [656]:
# Define the data directory
data_dir = "/data/personal/Alex_Chen/Torus/Data_vert_B0.6_pair/"

# Check what files are available
import glob
fld_files = sorted(glob.glob(os.path.join(data_dir, "fld.*.h5")))[:5]
ptc_files = sorted(glob.glob(os.path.join(data_dir, "ptc.*.h5")))[:5]

print(f"Data directory: {data_dir}")
print(f"\nFirst 5 field files:")
for f in fld_files:
    print(f"  {os.path.basename(f)}")
print(f"\nFirst 5 particle files:")
for f in ptc_files:
    print(f"  {os.path.basename(f)}")

# Check for grid and config files
print(f"\nGrid file exists: {os.path.exists(os.path.join(data_dir, 'grid.h5'))}")
print(f"Config file exists: {os.path.exists(os.path.join(data_dir, 'config.toml'))}")

Data directory: /data/personal/Alex_Chen/Torus/Data_vert_B0.6_pair/

First 5 field files:
  fld.00000.h5
  fld.00001.h5
  fld.00002.h5
  fld.00003.h5
  fld.00004.h5

First 5 particle files:
  ptc.00000.h5
  ptc.00001.h5
  ptc.00002.h5
  ptc.00003.h5
  ptc.00004.h5

Grid file exists: True
Config file exists: True


## Load Simulation Data with `DataKerrSchild`

We use the `DataKerrSchild` class from `datalib_gr_ks.py` to load the simulation data. This class:
- Reads the configuration from `config.toml`
- Loads the grid from `grid.h5`
- Sets up Kerr-Schild metric quantities
- Provides lazy loading of field and particle variables

In [657]:
# Load the simulation data
print("Loading data... this may take a moment")
data = ks.DataKerrSchild(data_dir)

print(f"\nData loaded successfully!")
print(f"Black hole spin (a): {data.a}")
print(f"Outer horizon radius (rH): {data.rH}")
print(f"Number of field steps: {len(data.fld_steps)}")
print(f"Field steps range: {data.fld_steps[0]} to {data.fld_steps[-1]}")
print(f"Number of particle steps: {len(data.ptc_steps)}")
print(f"Particle steps range: {data.ptc_steps[0]} to {data.ptc_steps[-1]}")

# Show configuration parameters
print(f"\nConfiguration keys:")
for key in data.conf.keys():
    print(f"  {key}: {data.conf[key]}")

Loading data... this may take a moment


/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder/datalib_gr_ks.py:79: RuntimeWarning: divide by zero encountered in divide
  return 1.0/Sigma(r,th,a)/np.sin(th)**2



Data loaded successfully!
Black hole spin (a): 0.998
Outer horizon radius (rH): 1.0632139225171164
Number of field steps: 2649
Field steps range: 0 to 2648
Number of particle steps: 2649
Particle steps range: 0 to 2648

Configuration keys:
  log_level: 0
  dt: 0.00025
  max_steps: 12000000
  fld_output_interval: 4000
  ptc_output_interval: 4000
  snapshot_interval: 40000
  rho_interval: 1
  max_ptc_num: 300000000
  max_ph_num: 100000000
  ptc_buffer_size: 10000000
  ph_buffer_size: 10000000
  ptc_segment_size: 20000000
  ph_segment_size: 20000000
  max_tracked_num: 4000000
  current_smoothing: 5
  tracked_fraction: 0.01
  q_e: 1.0
  bh_spin: 0.998
  use_implicit: True
  implicit_beta: 0.53
  ranks: [1, 4]
  N: [4096, 4096]
  size: [4.45, 3.141592654]
  lower: [0.0, 0.0]
  guard: [2, 2]
  periodic_boundary: [False, False]
  damping_length: 64
  damping_coef: 0.005
  damping_length_horizon: 16
  damping_coef_horizon: 0.01
  damp_to_background: True
  downsample: 2
  output_dir: Data_ver

## Access and Inspect Available Variables

The `DataKerrSchild` class provides access to field data, particle data, mesh data, and computed quantities.

In [658]:
# List all available variable keys
print("Available field keys:")
print(data._fld_keys)

print("\nAvailable particle keys:")
print(data._ptc_keys)

print("\nAvailable mesh keys:")
print(data._mesh_keys)

# Access a field variable (lazy loading - only loads when accessed)
print("\n--- Accessing field variables ---")
print(f"Rho_e shape: {data.Rho_e.shape}")
print(f"Rho_p shape: {data.Rho_p.shape}")
print(f"E1 shape: {data.E1.shape}")
print(f"B1 shape: {data.B1.shape}")

# Access computed quantities
print("\n--- Accessing computed quantities ---")
print(f"plasma_beta shape: {data.plasma_beta.shape}")
print(f"n_proper shape: {data.n_proper.shape}")

# Access mesh coordinates
print("\n--- Mesh coordinates ---")
print(f"x1 (R = r*sin(theta)) shape: {data.x1.shape}")
print(f"x2 (z = r*cos(theta)) shape: {data.x2.shape}")
print(f"_rv (r) shape: {data._rv.shape}")
print(f"_thetav (theta) shape: {data._thetav.shape}")

Available field keys:
['B1', 'B2', 'B3', 'B_sqr', 'Bmag', 'DdotB', 'E1', 'E2', 'E3', 'E_sqr', 'J01', 'J02', 'J03', 'J1', 'J2', 'J3', 'Jmag', 'Rho_e', 'Rho_p', 'Rho_ph', 'Rho_total', 'auxE1', 'auxE2', 'auxE3', 'auxH1', 'auxH2', 'auxH3', 'divB', 'divE', 'flux', 'flux_e1', 'flux_e2', 'flux_e3', 'flux_p1', 'flux_p2', 'flux_p3', 'flux_ph1', 'flux_ph2', 'flux_ph3', 'num_e', 'num_p', 'num_ph', 'pair_produced', 'photon_produced', 'step', 'stress_e00', 'stress_e01', 'stress_e02', 'stress_e03', 'stress_e11', 'stress_e12', 'stress_e13', 'stress_e22', 'stress_e23', 'stress_e33', 'stress_p00', 'stress_p01', 'stress_p02', 'stress_p03', 'stress_p11', 'stress_p12', 'stress_p13', 'stress_p22', 'stress_p23', 'stress_p33', 'stress_ph00', 'stress_ph01', 'stress_ph02', 'stress_ph03', 'stress_ph11', 'stress_ph12', 'stress_ph13', 'stress_ph22', 'stress_ph23', 'stress_ph33', 'time', 'fluxB', 'Dd1', 'Dd2', 'Dd3', 'D', 'Bd1', 'Bd2', 'Bd3', 'B', 'Ed1', 'Ed2', 'Ed3', 'Hd1', 'Hd2', 'Hd3', 'sigma', 'flux_upper', 'f

/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder/datalib_gr_ks.py:318: RuntimeWarning: invalid value encountered in divide
  b1 = -u_lower[...,0] * B[...,1] / alpha_val - (u_lower[...,2] * E[...,3] - u_lower[...,3] * E[...,2]) / alpha_val / sqrt_gm
/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder/datalib_gr_ks.py:319: RuntimeWarning: invalid value encountered in divide
  b2 = -u_lower[...,0] * B[...,2] / alpha_val - (u_lower[...,3] * E[...,1] - u_lower[...,1] * E[...,3]) / alpha_val / sqrt_gm
/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder/datalib_gr_ks.py:320: RuntimeWarning: invalid value encountered in divide
  b3 = -u_lower[...,0] * B[...,3] / alpha_val - (u_lower[...,1] * E[...,2] - u_lower[...,2] * E[...,1]) / alpha_val / sqrt_gm
/home/aaron/Research/Aperture4/problems/gr_2d_kerr_schild/analysis_folder/datalib_gr_ks.py:345: RuntimeWarning: invalid value encountered in divide
  b_upper = self.frf_B / bnorm

plasma_beta shape: (2048, 2048)
n_proper shape: (2048, 2048)

--- Mesh coordinates ---
x1 (R = r*sin(theta)) shape: (2048, 2048)
x2 (z = r*cos(theta)) shape: (2048, 2048)
_rv (r) shape: (2048, 2048)
_thetav (theta) shape: (2048, 2048)


## Extract Fields into NumPy Arrays

You can extract any field variable into a NumPy array for further analysis or custom visualization.

In [659]:
# Extract specific fields into numpy arrays
rho_e = data.Rho_e
rho_p = data.Rho_p
E1 = data.E1
B1 = data.B1
B2 = data.B2
B3 = data.B3

# Computed quantities
beta = data.plasma_beta
n_proper = data.n_proper

# Get coordinates
R = data.x1  # cylindrical radius = r * sin(theta)
z = data.x2  # height = r * cos(theta)

print("Extracted arrays:")
print(f"  rho_e: shape={rho_e.shape}, min={np.min(rho_e):.3e}, max={np.max(rho_e):.3e}")
print(f"  rho_p: shape={rho_p.shape}, min={np.min(rho_p):.3e}, max={np.max(rho_p):.3e}")
print(f"  E1: shape={E1.shape}, min={np.min(E1):.3e}, max={np.max(E1):.3e}")
print(f"  B1: shape={B1.shape}, min={np.min(B1):.3e}, max={np.max(B1):.3e}")
print(f"  plasma_beta: shape={beta.shape}, min={np.min(beta):.3e}, max={np.max(beta):.3e}")
print(f"  n_proper: shape={n_proper.shape}, min={np.min(n_proper):.3e}, max={np.max(n_proper):.3e}")

# You can also compute your own quantities
B_magnitude = np.sqrt(B1**2 + B2**2 + B3**2)
print(f"\nCustom computed B_magnitude: shape={B_magnitude.shape}, min={np.min(B_magnitude):.3e}, max={np.max(B_magnitude):.3e}")

Extracted arrays:
  rho_e: shape=(2048, 2048), min=-1.175e+03, max=0.000e+00
  rho_p: shape=(2048, 2048), min=0.000e+00, max=1.174e+03
  E1: shape=(2048, 2048), min=-1.102e-01, max=9.723e-02
  B1: shape=(2048, 2048), min=-5.931e-01, max=5.931e-01
  plasma_beta: shape=(2048, 2048), min=nan, max=nan
  n_proper: shape=(2048, 2048), min=0.000e+00, max=2.191e+03

Custom computed B_magnitude: shape=(2048, 2048), min=6.949e-03, max=6.898e-01


## Load Different Time Steps

You can load different time steps using `load_fld()` for field data and `load_ptc()` for particle data.

In [660]:
# Example: Load a different field step
step_to_load = data.fld_steps[10]  # Load the 11th step
print(f"Loading field step: {step_to_load}")
data.load_fld(step_to_load)

# Now data.Rho_e etc. will return values from the newly loaded step
rho_e_new = data.Rho_e
print(f"New rho_e shape: {rho_e_new.shape}")
print(f"New rho_e min: {np.min(rho_e_new):.3e}, max: {np.max(rho_e_new):.3e}")

# Load back the first step
data.load_fld(data.fld_steps[0])
print(f"\nReloaded initial step: {data.fld_steps[0]}")

# Time series example: get a time series of a field quantity
print("\n--- Time series example ---")
print("To get a time series, you can use data.time_series_fld('Rho_e')")
print("This returns a list of arrays for each field step.")
print(f"Total field steps available: {len(data.fld_steps)}")
print(f"Steps: {data.fld_steps[:5]} ... {data.fld_steps[-5:]}")

Loading field step: 10
New rho_e shape: (2048, 2048)
New rho_e min: -1.362e+03, max: 0.000e+00

Reloaded initial step: 0

--- Time series example ---
To get a time series, you can use data.time_series_fld('Rho_e')
This returns a list of arrays for each field step.
Total field steps available: 2649
Steps: [0, 1, 2, 3, 4] ... [2644, 2645, 2646, 2647, 2648]
